In [1]:
!pip install langchain langchain-core langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.7 MB/s eta 0:00:00


In [3]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

In [4]:
# ----------------------------
# API Key
# ----------------------------

#os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [5]:
# ----------------------------
# LLM
# ----------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


# ----------------------------
# Calculator Tool
# ----------------------------

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression like 2+2 or 10*5."""
    try:
        return str(eval(expression))
    except:
        return "Invalid expression"


# ----------------------------
# Prompt
# ----------------------------

prompt = ChatPromptTemplate.from_template(
"""
You are a helpful assistant.

If the user asks a math question, extract the expression and solve it using the calculator tool.

User: {input}
"""
)


# ----------------------------
# Simple Logic (Manual Tool Use)
# ----------------------------

def run_agent(user_input):

    # detect math
    if any(op in user_input for op in ["+", "-", "*", "/"]):
        result = calculator.invoke(user_input)
        return f"Answer: {result}"

    # otherwise use LLM
    response = llm.invoke(prompt.format(input=user_input))
    return response.content




In [6]:
# ----------------------------
# Chat Loop
# ----------------------------

while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        break

    answer = run_agent(query)

    print("\nAgent:", answer)


You: 25 * 3 

Agent: Answer: 75

You: exit
